# Euler shadows

Finite Euler products as densities, and the non-Radon direct limit on the length side.

**Claim evidenced.** Two claims from the essay.

1. The finite Euler shadow
$F_S(u) = \prod_{p \in S} P_{p^{-1/2}}(u \log p)$, a product of Poisson kernels
$P_a(\theta) = (1-a^2)/|1 - a e^{-i\theta}|^2$ sampled along the line
$\theta = u \log p$, coincides with
$\zeta_S(1)^{-1}\,|\zeta_S(1/2 + iu)|^2$, where
$\zeta_S(s) = \prod_{p\in S}(1 - p^{-s})^{-1}$. We verify the identity pointwise.

2. The direct length-side limit is **non-Radon**: the masses
$\kappa\!\left(\frac{n+1}{n}\right) = \frac{1}{\sqrt{n(n+1)}}$ have divergent
partial sums, so mass accumulates at $0$ and no locally finite limit measure exists.

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sympy import primerange


def euler_shadow(u: np.ndarray, primes) -> np.ndarray:
    """F_S(u) = prod_{p in S} P_{p^{-1/2}}(u log p)."""
    F = np.ones_like(u)
    for p in primes:
        a = p ** -0.5
        F *= (1 - a * a) / np.abs(1 - a * np.exp(-1j * u * math.log(p))) ** 2
    return F


u = np.linspace(-30, 30, 4001)
bounds = (10, 30, 100)

fig, ax = plt.subplots(figsize=(10, 4))
for bound in bounds:
    primes = list(primerange(2, bound))
    ax.plot(u, euler_shadow(u, primes), lw=1, label=f"S = primes < {bound}")
ax.set_xlabel("u (log-scale frequency)")
ax.set_ylabel(r"$F_S(u)$")
ax.set_title(r"Finite Euler shadows $F_S(u)=\prod_{p\in S}P_{p^{-1/2}}(u\log p)$")
ax.legend()
fig.tight_layout()
plt.show()

<Figure size 1000x400 with 1 Axes>

In [2]:
# Pointwise identity check: F_S(u) = zeta_S(1)^{-1} |zeta_S(1/2 + iu)|^2.
worst_abs = 0.0
worst_rel = 0.0
for bound in bounds:
    primes = list(primerange(2, bound))
    F = euler_shadow(u, primes)
    zeta_S_1_inv = np.prod([1.0 - 1.0 / p for p in primes])
    zeta_half = np.ones_like(u, dtype=complex)
    for p in primes:
        zeta_half /= 1.0 - p ** (-0.5 - 1j * u)
    rhs = zeta_S_1_inv * np.abs(zeta_half) ** 2
    abs_err = float(np.max(np.abs(F - rhs)))
    rel_err = float(np.max(np.abs(F - rhs) / rhs))
    worst_abs = max(worst_abs, abs_err)
    worst_rel = max(worst_rel, rel_err)
    print(f"primes < {bound:>3}: max abs error = {abs_err:.3e}   max rel error = {rel_err:.3e}")
print(f"\nworst over all S: max abs error = {worst_abs:.3e}, max rel error = {worst_rel:.3e}")
# F is large near u = 0 (it previews the pole of |zeta|^2), so the absolute
# error scales with F; the relative error is the machine-precision statement.
assert worst_rel < 1e-12

primes <  10: max abs error = 1.990e-13   max rel error = 2.508e-15
primes <  30: max abs error = 5.002e-12   max rel error = 3.606e-15
primes < 100: max abs error = 2.619e-10   max rel error = 5.076e-15

worst over all S: max abs error = 2.619e-10, max rel error = 5.076e-15


## The length side: a non-Radon limit

Partial sums of $\kappa\!\left(\frac{n+1}{n}\right) = \frac{1}{\sqrt{n(n+1)}}
\sim \frac1n$ up to $n = 2\cdot 10^5$.

In [3]:
n = np.arange(1, 200001)
kappa = 1.0 / np.sqrt(n * (n + 1.0))
partial = np.cumsum(kappa)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(n, partial)
ax.set_xlabel("n")
ax.set_ylabel(r"$\sum_{m\leq n}\kappa\left(\frac{m+1}{m}\right)$")
ax.set_title("Mass accumulating at 0: divergent partial sums (log axis)")
fig.tight_layout()
plt.show()

print(f"partial sum at n = 2*10^5: {partial[-1]:.4f}")
print(f"log(2*10^5)            : {math.log(200000):.4f}")

<Figure size 700x400 with 1 Axes>

partial sum at n = 2*10^5: 12.2251
log(2*10^5)            : 12.2061


**Interpretation.** The identity
$F_S(u) = \zeta_S(1)^{-1}|\zeta_S(1/2+iu)|^2$ holds to machine precision
(errors $\sim 10^{-13}$) for every finite prime set tested — as it must, since it
is an exact algebraic identity: with $a = p^{-1/2}$ one has $a^2 = 1/p$ and
$a e^{-iu\log p} = p^{-1/2-iu}$, so each Poisson factor is literally
$(1-p^{-1})\,|1-p^{-1/2-iu}|^{-2}$, the $p$-th factor of the right-hand side.
The shadows sharpen around
$u = 0$ as $S$ grows, previewing the blow-up of $|\zeta|^2$ at the pole.
On the length side, the partial sums of $\kappa((n+1)/n)$ track $\log n$ and
grow without bound: the straight line on the log axis is the numerical signature
that the direct limit measure assigns infinite mass to every neighborhood of $0$,
i.e. it is not Radon. Finite computation illustrates, but does not prove, the
divergence — that is the classical harmonic-series comparison.